In [2]:
import pandas as pd
from evidently import Report
from evidently.presets import DataDriftPreset
import numpy as np
import joblib


In [3]:
X_train = np.load("data/X_train.npy")
X_test = np.load("data/X_test.npy")
scaler = joblib.load("models/scaler.joblib") 

X_train_original = scaler.inverse_transform(X_train)
X_test_original = scaler.inverse_transform(X_test)

In [4]:
feature_names = [
    "daily_screen_time_hours",
    "phone_usage_before_sleep_minutes", 
    "sleep_duration_hours",
    "caffeine_intake_cups",
    "physical_activity_minutes",
    "notifications_received_per_day",
    "age",
    "screen_to_sleep_ratio"
]

reference = pd.DataFrame(X_train_original, columns=feature_names)
current = pd.DataFrame(X_test_original, columns=feature_names)

In [11]:
report = Report(metrics=[DataDriftPreset()])
result = report.run(reference_data=reference, current_data=current)
result.save_html("monitoring_report.html")

In [10]:
print(type(result))
print(dir(result))

<class 'evidently.core.report.Snapshot'>
['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_context', '_metadata', '_metrics', '_name', '_report', '_repr_html_', '_run_items', '_snapshot_item', '_tags', '_tests_widgets', '_timestamp', '_to_v1', '_top_level_metrics', '_widgets', 'context', 'dict', 'dump_dict', 'dumps', 'get_html_str', 'get_name', 'json', 'load', 'load_dict', 'load_model', 'loads', 'metric_results', 'render_only_fingerprint', 'report', 'run', 'save_html', 'save_json', 'set_name', 'tests_results', 'to_snapshot_model']


In [12]:
# Simuler des nouvelles données avec drift
current_drift = reference.copy()
current_drift["daily_screen_time_hours"] += 3  # les gens passent 3h de plus sur écran
current_drift["sleep_duration_hours"] -= 1      # ils dorment 1h de moins

result_drift = report.run(reference_data=reference, current_data=current_drift)
result_drift.save_html("monitoring_report_drift.html")